# Lesson 5: Structured Output

## WHY
In Lesson 4, your moderation agent returned a free-text verdict like *"I've muted user_123 for spam..."*.
That's fine for a human reading Discord, but terrible for code that needs to act on it.

How do you programmatically check the action taken? `if "mute" in response`? That's brittle — the model might say "timeout" instead of "mute", or bury the action mid-paragraph.

**Structured output** forces the LLM to return data in a validated schema — a Pydantic model.
Instead of parsing free text, you get a Python object with typed fields you can trust:

```python
verdict.action      # "mute" — guaranteed to be one of ["allow", "warn", "mute", "kick", "ban"]
verdict.confidence   # 0.92 — a float, not the string "high"
verdict.reasoning    # "Repeat offender + phishing link..."
```

This is essential in production because:
- **Downstream code** can branch on `verdict.action` with zero parsing
- **Validation** catches hallucinated values (e.g., action="delete" when only allow/warn/mute/kick/ban are valid)
- **Logging and analytics** get clean, consistent records
- **Testing** becomes straightforward — assert on fields, not regex on prose

**By the end of this notebook you will:**
1. Define Pydantic models that describe the output you want from an LLM
2. Use `.with_structured_output()` on a chat model for standalone structured calls
3. Use `response_format=` on `create_agent` for structured output from agents with tools
4. Understand when to use each approach
5. Build production-ready moderation verdict models

## Setup

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
print("API key loaded:", "OPENAI_API_KEY" in os.environ)

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## WHAT — Structured Output

Structured output means the LLM returns a **JSON object that conforms to a schema you define**, instead of free-form text.

LangChain gives you two ways to do this:

| Approach | When to use | How |
|----------|-------------|-----|
| `.with_structured_output()` | Standalone LLM call — no tools, no agent loop | Call it on any chat model |
| `response_format=` on `create_agent` | Agent with tools that should return structured data after its ReAct loop | Pass a Pydantic model to the agent factory |

Both approaches use the same Pydantic models. The difference is *where* in the pipeline the structure is enforced.

📖 [Structured output conceptual guide](https://python.langchain.com/docs/concepts/structured_outputs/)

## WHAT — Pydantic Models as Output Schemas

You already used Pydantic in Lesson 3 for tool *input* schemas. Now you'll use it for LLM *output* schemas.
Same library, different direction — instead of telling the LLM what arguments to pass *in*, you're telling it what fields to return *out*.

Key rules for output schemas:
- Every field needs a `description` — the LLM reads it to understand what to put there
- Use `Literal` for enum-like fields to restrict values (e.g., `Literal["allow", "warn", "ban"]`)
- Keep models flat when possible — nested models work but add complexity
- Field names matter — descriptive names help the LLM fill them correctly

## HOW — Defining a Moderation Verdict Model

Let's define a Pydantic model that captures everything we'd want from a moderation decision.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field


class ModerationVerdict(BaseModel):
    """A structured moderation decision for a Discord message."""

    action: Literal["allow", "warn", "mute", "kick", "ban"] = Field(
        description="The moderation action to take on the message"
    )
    reasoning: str = Field(
        description="Step-by-step explanation of why this action was chosen"
    )
    violated_rule: str | None = Field(
        default=None,
        description="Which specific server rule was violated, if any"
    )
    confidence: float = Field(
        description="Confidence in the decision from 0.0 (uncertain) to 1.0 (certain)"
    )
    duration_minutes: int = Field(
        default=0,
        description="Duration in minutes for temporary actions (mute). 0 for non-temporary actions."
    )


# Inspect the schema the LLM will see
import json
print(json.dumps(ModerationVerdict.model_json_schema(), indent=2))

Notice how the schema includes:
- The `enum` constraint on `action` — the LLM can only pick from the allowed values
- `description` on every field — this is the LLM's instructions for what to put there
- `default` values for optional fields
- Type information (`string`, `number`, `null`) that the LLM respects

This schema gets sent to the LLM as part of the API call. The LLM provider (OpenAI, Anthropic) enforces it at the token generation level — the model literally *cannot* return invalid JSON.

## HOW — `.with_structured_output()` (Standalone)

The simplest way to get structured output is calling `.with_structured_output()` on a chat model.
This wraps the model so that `.invoke()` returns a Pydantic object instead of an `AIMessage`.

```python
structured_llm = llm.with_structured_output(MyModel)
result = structured_llm.invoke("some prompt")  # returns MyModel instance, not AIMessage
```

Use this when you have a **single LLM call** with no tool use — just "here's some text, give me structured data back."

📖 [.with_structured_output() reference](https://python.langchain.com/docs/how_to/structured_output/)

In [ ]:
# Wrap the model for structured output
structured_llm = llm.with_structured_output(ModerationVerdict)

print(f"Type: {type(structured_llm)}")
print("Ready — .invoke() will now return a ModerationVerdict")

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

# Single-call moderation: no tools, just classification
verdict = structured_llm.invoke([
    SystemMessage(content=(
        "You are a Discord moderation classifier.\n"
        "Evaluate messages for rule violations and return a structured verdict.\n"
        "Server rules: no spam, no phishing, no harassment, be respectful."
    )),
    HumanMessage(content=(
        "User posted: 'FREE DISCORD NITRO! Click: totally-not-a-scam.com'"
    )),
])

# The result is a Pydantic object — not an AIMessage!
print(f"Type: {type(verdict)}")
print(f"Action: {verdict.action}")
print(f"Confidence: {verdict.confidence}")
print(f"Violated rule: {verdict.violated_rule}")
print(f"Duration: {verdict.duration_minutes} min")
print(f"Reasoning: {verdict.reasoning}")

### What Just Happened

1. We called `.invoke()` with messages — same as Lessons 1-2
2. Instead of getting an `AIMessage` with `.content` (free text), we got a `ModerationVerdict` instance
3. Every field is typed and validated — `action` is guaranteed to be one of the allowed values
4. We can branch on `verdict.action` directly in code — no string parsing needed

This is the power of structured output: **the LLM's response is code-friendly from the start**.

In [ ]:
# Programmatic branching — this is why structured output matters
def handle_verdict(verdict: ModerationVerdict) -> str:
    """Take action based on a structured moderation verdict.

    Parameters
    ----------
    verdict : ModerationVerdict
        The structured verdict from the moderation model.

    Returns
    -------
    str
        Description of the action taken.
    """
    match verdict.action:
        case "allow":
            return "Message allowed — no action needed"
        case "warn":
            return f"Warning issued: {verdict.reasoning}"
        case "mute":
            return f"Muted for {verdict.duration_minutes}m: {verdict.reasoning}"
        case "kick":
            return f"Kicked: {verdict.reasoning}"
        case "ban":
            return f"Banned: {verdict.reasoning}"


print(handle_verdict(verdict))

In [ ]:
# Test with a harmless message
verdict_safe = structured_llm.invoke([
    SystemMessage(content=(
        "You are a Discord moderation classifier.\n"
        "Evaluate messages for rule violations and return a structured verdict.\n"
        "Server rules: no spam, no phishing, no harassment, be respectful."
    )),
    HumanMessage(content=(
        "User posted: 'Hey does anyone know a good Python tutorial?'"
    )),
])

print(f"Action: {verdict_safe.action}")
print(f"Confidence: {verdict_safe.confidence}")
print(f"Reasoning: {verdict_safe.reasoning}")
print(f"\nHandle: {handle_verdict(verdict_safe)}")

### Validation in Action

What if we try to create an invalid `ModerationVerdict` manually? Pydantic catches it.

In [ ]:
from pydantic import ValidationError

# This will fail — "delete" is not a valid action
try:
    bad_verdict = ModerationVerdict(
        action="delete",  # not in the Literal list!
        reasoning="Testing",
        confidence=0.5,
    )
except ValidationError as e:
    print("Validation error caught!")
    print(e)

print("\n→ The LLM provider enforces this at generation time,")
print("  so you won't normally see this. But it's a safety net.")

## HOW — `include_raw=True` (Getting the AIMessage Too)

Sometimes you want both the structured output AND the raw `AIMessage` (for logging, token counts, etc.).
Pass `include_raw=True` to get a dict with both.

In [ ]:
# Get both the parsed object and the raw AIMessage
structured_llm_raw = llm.with_structured_output(ModerationVerdict, include_raw=True)

raw_result = structured_llm_raw.invoke([
    SystemMessage(content="Evaluate Discord messages. Rules: no spam, no harassment."),
    HumanMessage(content="User posted: 'You're all idiots and I hope you get banned'"),
])

print(f"Keys: {list(raw_result.keys())}")
print(f"\nParsed type: {type(raw_result['parsed'])}")
print(f"Parsed action: {raw_result['parsed'].action}")
print(f"\nRaw type: {type(raw_result['raw'])}")
print(f"Raw usage: {raw_result['raw'].usage_metadata}")

The dict contains:
- `parsed` — your Pydantic model instance
- `raw` — the original `AIMessage` with token usage, model info, etc.
- `parsing_error` — `None` if parsing succeeded, or the error if it failed

This is useful in production for cost tracking (token counts) and debugging.

## HOW — Structured Output with Agents (`response_format=`)

`.with_structured_output()` is great for one-shot classification, but what about agents that use tools?
You can't use `.with_structured_output()` on an agent — the agent needs free-form `AIMessage` responses during its ReAct loop to make tool calls.

Instead, `create_agent` has a `response_format=` parameter. It works like this:
1. The agent runs its normal ReAct loop (reasoning, tool calls, etc.)
2. After the agent decides it's done, it makes **one final LLM call** to format its conclusion into your schema
3. The structured result is stored in `state["structured_response"]`

This gives you the best of both worlds: tool use during reasoning, structured data as the final output.

```python
from langchain.agents import create_agent

agent = create_agent(
    "openai:gpt-4o-mini",
    tools=my_tools,
    system_prompt="...",
    response_format=MyPydanticModel,  # <-- structured final output
)

result = agent.invoke({"messages": [...]})
result["structured_response"]  # MyPydanticModel instance
```

In [ ]:
from langchain_core.tools import tool
from pydantic import BaseModel, Field


# Redefine moderation tools (self-contained notebook)
SERVER_RULES = {
    "general": ["Be respectful", "Stay on-topic", "No excessive caps"],
    "spam": ["No self-promo", "No repeated messages", "No phishing links"],
    "nsfw": ["No NSFW outside designated channels", "No gore or shock content"],
}

WARN_HISTORY = {
    "user_123": [
        {"reason": "Spam links in #general", "date": "2026-03-15"},
        {"reason": "Harassment", "date": "2026-04-01"},
    ],
    "user_456": [{"reason": "NSFW in wrong channel", "date": "2026-02-20"}],
    "user_789": [],
}


@tool
def get_server_rules(category: str = "all") -> str:
    """Look up the Discord server's moderation rules.

    Parameters
    ----------
    category : str
        Rule category: 'general', 'spam', 'nsfw', or 'all'.
    """
    if category == "all":
        return "\n".join(f"{c}: {', '.join(r)}" for c, r in SERVER_RULES.items())
    rules = SERVER_RULES.get(category)
    if rules:
        return "\n".join(f"{i}. {r}" for i, r in enumerate(rules, 1))
    return f"Unknown category: {category}. Available: {list(SERVER_RULES.keys())}"


class UserLookupInput(BaseModel):
    """Input for user history lookup."""
    user_id: str = Field(description="The Discord user's ID")


@tool(args_schema=UserLookupInput)
def get_user_warn_history(user_id: str) -> str:
    """Look up a user's previous warnings. Check before deciding on action."""
    if user_id not in WARN_HISTORY:
        return f"No record found for user {user_id}"
    warnings = WARN_HISTORY[user_id]
    if not warnings:
        return f"User {user_id} has a clean record (0 warnings)"
    lines = [f"User {user_id}: {len(warnings)} warning(s)"]
    for w in warnings:
        lines.append(f"  - {w['date']}: {w['reason']}")
    return "\n".join(lines)


class ModActionInput(BaseModel):
    """Input for moderation actions."""
    user_id: str = Field(description="Discord user ID")
    action: str = Field(description="Action: 'warn', 'mute', 'kick', or 'ban'")
    reason: str = Field(description="Reason for the action")
    duration_minutes: int = Field(default=0, description="Duration for temp actions. 0 = permanent.")


@tool(args_schema=ModActionInput)
def issue_moderation_action(
    user_id: str, action: str, reason: str, duration_minutes: int = 0
) -> str:
    """Issue a moderation action against a user. Check rules and history FIRST."""
    valid = ["warn", "mute", "kick", "ban"]
    if action not in valid:
        return f"Invalid action. Must be one of: {valid}"
    dur = f" for {duration_minutes}m" if duration_minutes else ""
    return f"[SIM] {action.upper()}{dur} on {user_id}: {reason}"


moderation_tools = [get_server_rules, get_user_warn_history, issue_moderation_action]
print(f"Tools defined: {[t.name for t in moderation_tools]}")

In [ ]:
from langchain.agents import create_agent

# Create an agent with structured output
moderation_agent = create_agent(
    "openai:gpt-4o-mini",
    tools=moderation_tools,
    system_prompt=(
        "You are a Discord server moderation bot.\n"
        "WORKFLOW:\n"
        "1. Check the relevant server rules\n"
        "2. Check the user's warning history\n"
        "3. Decide on an appropriate action\n"
        "4. Issue the action if needed\n\n"
        "GUIDELINES:\n"
        "- First offense + minor violation → warn\n"
        "- Repeat offender + minor violation → mute (30 min)\n"
        "- Severe violation (phishing, threats) → ban regardless of history\n"
        "- Always explain your reasoning"
    ),
    response_format=ModerationVerdict,  # <-- the key addition
)

print(f"Agent type: {type(moderation_agent)}")
print("Agent created with response_format=ModerationVerdict")

In [ ]:
from langchain_core.messages import HumanMessage

# Test: Repeat offender + phishing
result = moderation_agent.invoke({
    "messages": [
        HumanMessage(
            content=(
                "User user_123 posted in #general: "
                "'FREE DISCORD NITRO! Click: totally-not-a-scam.com'\n"
                "Evaluate and take action."
            )
        )
    ]
})

# The structured response lives in state["structured_response"]
verdict = result["structured_response"]

print(f"Type: {type(verdict)}")
print(f"Action: {verdict.action}")
print(f"Confidence: {verdict.confidence}")
print(f"Violated rule: {verdict.violated_rule}")
print(f"Duration: {verdict.duration_minutes} min")
print(f"Reasoning: {verdict.reasoning}")

### What Happened Under the Hood

1. The agent ran its ReAct loop as normal — checking rules, looking up history, issuing action
2. After the last tool call, the agent made a **final LLM call** to format its conclusion into the `ModerationVerdict` schema
3. The result was stored in `result["structured_response"]` as a Pydantic object

The agent's message history (`result["messages"]`) still has the full trace — the structured response is *additional* output, not a replacement.

In [ ]:
# The full message trace is still available
print("=== Agent Trace ===")
for i, msg in enumerate(result["messages"]):
    role = msg.__class__.__name__
    if role == "SystemMessage":
        print(f"[{i}] SYSTEM: (prompt omitted)")
    elif hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            args_str = ", ".join(f"{k}={v!r}" for k, v in tc["args"].items())
            print(f"[{i}] AGENT -> {tc['name']}({args_str})")
    elif role == "ToolMessage":
        content = msg.content[:80] + "..." if len(msg.content) > 80 else msg.content
        print(f"[{i}] TOOL  -> {content}")
    elif role == "HumanMessage":
        content = msg.content[:80] + "..." if len(msg.content) > 80 else msg.content
        print(f"[{i}] USER  -> {content}")
    else:
        content = msg.content[:80] + "..." if len(msg.content) > 80 else msg.content
        print(f"[{i}] AGENT -> {content}")

print(f"\n=== Structured Response ===")
print(f"Action: {verdict.action} (confidence: {verdict.confidence})")

In [ ]:
# Test: Clean user + harmless message
result_safe = moderation_agent.invoke({
    "messages": [
        HumanMessage(
            content=(
                "User user_789 posted in #general: "
                "'Hey can someone help me with a Python question?'\n"
                "Evaluate this message."
            )
        )
    ]
})

verdict_safe = result_safe["structured_response"]
print(f"Action: {verdict_safe.action}")
print(f"Confidence: {verdict_safe.confidence}")
print(f"Reasoning: {verdict_safe.reasoning}")
print(f"\nHandle: {handle_verdict(verdict_safe)}")

## Deep Dive — When to Use Which Approach

| Scenario | Approach | Why |
|----------|----------|-----|
| Classify a message (no tool use needed) | `.with_structured_output()` | Simpler, single LLM call, lower cost |
| Agent needs tools AND structured final output | `response_format=` on `create_agent` | Agent loop handles tools; structured output is the formatted conclusion |
| Quick triage before running the full agent | `.with_structured_output()` | Fast pre-filter: if `verdict.action == "allow"`, skip the agent entirely |
| Log every agent decision consistently | `response_format=` on `create_agent` | Guarantees every agent run produces a typed record |

A common **production pattern** combines both:
1. **Fast pass**: Use `.with_structured_output()` to cheaply classify the message
2. **If flagged**: Run the full agent (with tools for history lookup, rule checking) using `response_format=` for a structured verdict

This saves cost because most messages are harmless — you only pay for the full agent on flagged ones.

## HOW — Two-Stage Moderation Pipeline

Let's implement the production pattern: fast classification first, full agent only when needed.

In [ ]:
class QuickTriageVerdict(BaseModel):
    """A lightweight triage result — just flag or pass."""

    flagged: bool = Field(
        description="True if the message might violate server rules and needs full review"
    )
    reason: str = Field(
        description="Brief explanation of why the message was flagged or passed"
    )


triage_llm = llm.with_structured_output(QuickTriageVerdict)

triage_prompt = SystemMessage(content=(
    "You are a fast message triage system.\n"
    "Quickly decide if a message MIGHT violate Discord rules.\n"
    "When in doubt, flag it — a more thorough review will follow.\n"
    "Rules: no spam, no phishing, no harassment, be respectful."
))

print("Triage model ready")

In [ ]:
def moderate_message(user_id: str, message: str) -> ModerationVerdict | str:
    """Two-stage moderation: fast triage, then full agent if flagged.

    Parameters
    ----------
    user_id : str
        The Discord user ID who sent the message.
    message : str
        The message content to evaluate.

    Returns
    -------
    ModerationVerdict | str
        A structured verdict if the message was flagged, or a string "allowed" if it passed triage.
    """
    # Stage 1: Fast triage (cheap, no tools)
    triage = triage_llm.invoke([
        triage_prompt,
        HumanMessage(content=f"User {user_id} posted: '{message}'"),
    ])
    print(f"[Triage] Flagged: {triage.flagged} — {triage.reason}")

    if not triage.flagged:
        return "allowed"

    # Stage 2: Full agent with tools (only for flagged messages)
    print("[Agent] Running full moderation agent...")
    result = moderation_agent.invoke({
        "messages": [
            HumanMessage(
                content=(
                    f"User {user_id} posted in #general: '{message}'\n"
                    "Evaluate and take action."
                )
            )
        ]
    })
    return result["structured_response"]

In [ ]:
# Test 1: Harmless message — should stop at triage
result1 = moderate_message("user_789", "Hey does anyone know a good Python tutorial?")
print(f"Result: {result1}\n")

In [ ]:
# Test 2: Suspicious message — should trigger full agent
result2 = moderate_message("user_123", "FREE DISCORD NITRO! Click: totally-not-a-scam.com")
print(f"\nResult type: {type(result2)}")
if isinstance(result2, ModerationVerdict):
    print(f"Action: {result2.action}")
    print(f"Confidence: {result2.confidence}")
    print(f"Reasoning: {result2.reasoning}")

### Cost Comparison

- **Triage only** (harmless message): ~1 LLM call, ~100-200 tokens
- **Full agent** (flagged message): ~3-5 LLM calls (triage + agent reasoning + tool calls + structured formatting), ~1000-2000 tokens

If 90% of messages are harmless (typical for most servers), you save ~80% of your LLM cost by triaging first.

## HOW — Richer Pydantic Models

Pydantic models can be more sophisticated. Here are patterns you'll use in production.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field


class ViolationDetail(BaseModel):
    """Details about a specific rule violation."""

    category: Literal["spam", "harassment", "nsfw", "off_topic", "other"] = Field(
        description="The category of violation"
    )
    rule: str = Field(
        description="The specific rule that was violated"
    )
    severity: Literal["low", "medium", "high", "critical"] = Field(
        description="How severe the violation is"
    )


class DetailedModerationVerdict(BaseModel):
    """A detailed moderation verdict with violation breakdowns."""

    action: Literal["allow", "warn", "mute", "kick", "ban"] = Field(
        description="The moderation action to take"
    )
    reasoning: str = Field(
        description="Step-by-step explanation of the decision"
    )
    violations: list[ViolationDetail] = Field(
        default_factory=list,
        description="List of specific violations found. Empty if message is clean."
    )
    confidence: float = Field(
        description="Confidence in the decision from 0.0 to 1.0"
    )
    requires_human_review: bool = Field(
        default=False,
        description="True if the decision is uncertain and a human moderator should review"
    )
    duration_minutes: int = Field(
        default=0,
        description="Duration for temporary actions. 0 for non-temporary."
    )


# Inspect the nested schema
print(json.dumps(DetailedModerationVerdict.model_json_schema(), indent=2))

In [ ]:
# Test the detailed model with .with_structured_output()
detailed_llm = llm.with_structured_output(DetailedModerationVerdict)

detailed_verdict = detailed_llm.invoke([
    SystemMessage(content=(
        "You are a Discord moderation classifier.\n"
        "Evaluate messages thoroughly and identify all violations.\n"
        "Server rules: no spam, no phishing, no harassment, be respectful, stay on topic."
    )),
    HumanMessage(content=(
        "User posted: 'SHUT UP YOU IDIOT also check out my YouTube: youtube.com/myChannel FREE SUB4SUB'"
    )),
])

print(f"Action: {detailed_verdict.action}")
print(f"Confidence: {detailed_verdict.confidence}")
print(f"Needs human review: {detailed_verdict.requires_human_review}")
print(f"\nViolations ({len(detailed_verdict.violations)}):")
for v in detailed_verdict.violations:
    print(f"  [{v.severity.upper()}] {v.category}: {v.rule}")
print(f"\nReasoning: {detailed_verdict.reasoning}")

### Nested Models

The `violations` field is a `list[ViolationDetail]` — the LLM creates one entry per violation found.
This is powerful for messages that break *multiple* rules simultaneously.

Keep nesting shallow (1-2 levels). Deeply nested schemas are harder for the LLM to fill correctly.

## HOW — Structured Output with Model Strings

Both approaches work with model string IDs (no need to import `ChatOpenAI`).
This is relevant when you use `create_agent` with a string — the model is created internally.

In [ ]:
from langchain.chat_models import init_chat_model

# init_chat_model creates a model from a string — same as what create_agent does internally
string_llm = init_chat_model("openai:gpt-4o-mini", temperature=0)

# .with_structured_output() works on the resulting model
string_structured = string_llm.with_structured_output(ModerationVerdict)

verdict = string_structured.invoke([
    SystemMessage(content="Evaluate Discord messages. Rules: no spam, no harassment."),
    HumanMessage(content="User posted: 'Check out my Twitch stream! Follow for follow!'"),
])

print(f"Action: {verdict.action}")
print(f"Confidence: {verdict.confidence}")
print(f"Reasoning: {verdict.reasoning}")

## Importable Moderation Verdict Models

Here are the Pydantic models packaged for import into the bot codebase.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field


# ── Quick triage (for .with_structured_output()) ──────────────

class QuickTriageVerdict(BaseModel):
    """Lightweight triage result for fast message screening."""

    flagged: bool = Field(
        description="True if the message might violate rules and needs full review"
    )
    reason: str = Field(
        description="Brief explanation of why the message was flagged or passed"
    )


# ── Full moderation verdict (for agents with response_format=) ─

class ViolationDetail(BaseModel):
    """Details about a specific rule violation."""

    category: Literal["spam", "harassment", "nsfw", "off_topic", "other"] = Field(
        description="The category of violation"
    )
    rule: str = Field(
        description="The specific rule that was violated"
    )
    severity: Literal["low", "medium", "high", "critical"] = Field(
        description="How severe the violation is"
    )


class ModerationVerdict(BaseModel):
    """A structured moderation decision for a Discord message."""

    action: Literal["allow", "warn", "mute", "kick", "ban"] = Field(
        description="The moderation action to take on the message"
    )
    reasoning: str = Field(
        description="Step-by-step explanation of why this action was chosen"
    )
    violations: list[ViolationDetail] = Field(
        default_factory=list,
        description="Specific violations found. Empty if message is clean."
    )
    confidence: float = Field(
        description="Confidence in the decision from 0.0 to 1.0"
    )
    requires_human_review: bool = Field(
        default=False,
        description="True if uncertain and a human moderator should review"
    )
    duration_minutes: int = Field(
        default=0,
        description="Duration in minutes for temporary actions. 0 for non-temporary."
    )

## Summary

| Concept | Key takeaway |
|---------|-------------|
| Pydantic output models | Define the exact shape of data you want from the LLM — field descriptions are part of the prompt |
| `Literal` types | Constrain fields to specific values — the LLM cannot return anything else |
| `.with_structured_output()` | Wrap any chat model for single-call structured responses |
| `include_raw=True` | Get both the Pydantic object and the raw `AIMessage` (for logging/costs) |
| `response_format=` on `create_agent` | Structured output from agents — tools during reasoning, typed data as final output |
| `state["structured_response"]` | Where the agent's structured output lives in the result dict |
| Two-stage pipeline | Cheap triage first (`.with_structured_output()`), full agent only for flagged messages |
| Nested models | Use `list[SubModel]` for multi-violation detection — keep nesting shallow |
| `init_chat_model` | Creates a model from a string like `"openai:gpt-4o-mini"` — same as what `create_agent` uses internally |

**Next up → Module 4, Lesson 6:** Wire the moderation agent into the Discord bot's `on_message()` event — async patterns, error handling, and cost awareness.